Creating two clustering models using data curated by StatCan.

In [ ]:
!python -m pip install "numpy<2.0" "pandas<2.0"
!python -m pip install scikit-learn==1.6.0
!python -m pip install matplotlib==3.9.3
!python -m pip install hdbscan==0.8.40
!python -m pip install geopandas==1.0.1
!python -m pip install contextily==1.6.2
!python -m pip install shapely==2.0.6

In [5]:
import importlib.util as u
print({p: u.find_spec(p) is not None for p in ["numpy","pandas","sklearn","matplotlib", "shapely", "contextily", "geopandas", "hdbscan"]})

{'numpy': True, 'pandas': True, 'sklearn': True, 'matplotlib': True, 'shapely': True, 'contextily': True, 'geopandas': True, 'hdbscan': True}


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import hdbscan
from sklearn.preprocessing import StandardScaler

# geographical tools
import geopandas as gpd  # pandas dataframe-like geodataframes for geographical data
import contextily as ctx  # used for obtianing a basemap of Canada
from shapely.geometry import Point

import warnings
warnings.filterwarnings('ignore')

In [8]:
# downloading canada map

import requests
import zipfile
import io
import os

zip_file_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YcUk-ytgrPkmvZAh5bf7zA/Canada.zip'

# Directory to save the extracted TIFF file
output_dir = './'
os.makedirs(output_dir, exist_ok=True)

# Step 1: Download the ZIP file
response = requests.get(zip_file_url)
response.raise_for_status()  # Ensure the request was successful
# Step 2: Open the ZIP file in memory
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    # Step 3: iterating over the files in the ZIP
    for file_name in zip_ref.namelist():
        if file_name.endswith('.tif'):  # Check if it's a TIFF file
            # Step 4: extracting the TIFF file
            zip_ref.extract(file_name, output_dir)
            print(f"Downloaded and extracted: {file_name}")

Downloaded and extracted: Canada.tif


In [ ]:
# plotting my data points as geographic locations on top of a Canada map

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

def plot_clustered_locations(df, title='Museums Clustered by Proximity'):


    # Check required cols
    required_cols = {'Latitude', 'Longitude', 'Cluster'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")

    # Load coordinates into a GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
        crs="EPSG:4326"
    )

    # Reproject to Web Mercator
    gdf = gdf.to_crs(epsg=3857)

    # Create plot
    fig, ax = plt.subplots(figsize=(15, 10))

    # Separate clustered points from noise
    non_noise = gdf[gdf['Cluster'] != -1]
    noise = gdf[gdf['Cluster'] == -1]

    # Plot noise points
    if not noise.empty:
        noise.plot(ax=ax, color='k', markersize=30, edgecolor='r', alpha=1, label='Noise')

    # Plot clustered points
    if not non_noise.empty:
        non_noise.plot(
            ax=ax,
            column='Cluster',
            cmap='tab10',
            markersize=30,
            edgecolor='k',
            legend=False,
            alpha=0.6
        )

    # Add basemap
    ctx.add_basemap(ax, source='./Canada.tif')

    # Format
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()

In [13]:
# data
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/r-maSj5Yegvw2sJraT15FA/ODCAF-v1-0.csv'
df = pd.read_csv(url, encoding = "ISO-8859-1")

In [14]:
df.head()

,Index,Facility_Name,Source_Facility_Type,ODCAF_Facility_Type,Provider,Unit,Street_No,Street_Name,Postal_Code,City,Prov_Terr,Source_Format_Address,CSD_Name,CSDUID,PRUID,Latitude,Longitude
0,1,#Hashtag Gallery,..,gallery,toronto,..,801,dundas st w,M6J 1V2,toronto,on,801 dundas st w,Toronto,3520005,35,43.65169472,-79.40803272
1,2,'Ksan Historical Village & Museum,historic site-building or park,museum,canadian museums association,..,1500,62 hwy,V0J 1Y0,hazelton,bc,1500 hwy 62 hazelton british columbia v0j 1y0 ...,Hazelton,5949022,59,55.2645508,-127.6428124
2,3,'School Days' Museum,community/regional museum,museum,canadian museums association,..,427,queen st,E3B 5R6,fredericton,nb,427 queen st fredericton new brunswick e3b 5r6...,Fredericton,1310032,13,45.963283,-66.6419017
3,4,10 Austin Street,built heritage properties,heritage or historic site,moncton,..,10,austin st,E1C 1Z6,moncton,nb,10 austin st,Moncton,1307022,13,46.09247776,-64.78022946
4,5,10 Gates Dancing Inc.,arts,miscellaneous,ottawa,..,..,..,..,ottawa,on,..,Ottawa,3506008,35,45.40856224,-75.71536766


In [15]:
df.ODCAF_Facility_Type.value_counts()

library or archives                     3013
museum                                  1938
gallery                                  810
heritage or historic site                620
theatre/performance and concert hall     583
festival site                            346
miscellaneous                            343
art or cultural centre                   225
artist                                    94
Name: ODCAF_Facility_Type, dtype: int64

In [16]:
# only include museums
df = df[df.ODCAF_Facility_Type == 'museum']
df.ODCAF_Facility_Type.value_counts()

museum    1938
Name: ODCAF_Facility_Type, dtype: int64

In [ ]:
# selecting only latitude and longitude as input
# updating table
df = df[['Latitude', 'Longitude']]
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1938 entries, 1 to 7969
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Latitude   1938 non-null   object
 1   Longitude  1938 non-null   object
dtypes: object(2)
memory usage: 45.4+ KB


In [19]:
# delete any museum that does not have those coordinates
df = df[df.Latitude!='..']

# Convert to float
df[['Latitude','Longitude']] = df[['Latitude','Longitude']].astype('float')

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is an unsupervised machine learning algorithm that groups closely packed data points into clusters based on the density of points in a specific area.<br>
<br>
I should not use dataset-based normalization like z-score here, because latitude and longitude already have a known physical meaning and known global scales. And 1 degree of latitude and 1 degree of longitude are not being treated on the same numeric scale.

In [20]:
# Building a DBSCAN model
coords_scaled = df.copy()
coords_scaled["Latitude"] = 2*coords_scaled["Latitude"]

In [22]:
# Applying DBSCAN with Euclidean distance to the scaled coordinates
min_samples=3 # minimum number of samples needed to form a neighbourhood
eps=1.0 # neighbourhood search radius
metric='euclidean' # distance measure 

# now I have a fitted clustering result
dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric=metric).fit(coords_scaled)

dbscan contains:<br>
- which points belong to which cluster
- which points are noise/outliers
- the cluster label for each input point

In [27]:
# an array with one label per item of coords_scaled.
dbscan.labels_


array([0, 1, 2, ..., 4, 2, 2], dtype=int64)

In [26]:
dbscan.labels_.shape

(1607,)

The output showed point 1 belongs to cluster 0 and point 1 belongs to cluster 1, etc

In [24]:
df['Cluster'] = dbscan.fit_predict(coords_scaled)  # Assign the cluster labels

# Display the size of each cluster
df['Cluster'].value_counts()

 4     701
 2     192
 1     181
 7     134
 3      94
-1      79
 6      30
 10     27
 8      21
 11     15
 15     13
 20     11
 16     10
 19      9
 27      8
 12      7
 24      6
 18      6
 28      6
 26      6
 14      6
 5       6
 22      4
 9       4
 13      4
 30      3
 31      3
 29      3
 0       3
 25      3
 23      3
 21      3
 17      3
 32      3
Name: Cluster, dtype: int64